In [1]:
import pandas as pd

df = pd.read_csv('data-engineer-interview-data (2).csv')


In [2]:
df["RefersToDocumentNumber"] = pd.to_numeric(df["RefersToDocumentNumber"], errors="coerce")
df["RefersToDocumentYear"] = pd.to_numeric(df["RefersToDocumentYear"], errors="coerce")
df["DocumentNumber"] = pd.to_numeric(df["DocumentNumber"], errors="coerce")


In [3]:
mask = (
    df["RefersToDocumentNumber"].between(1900, 2100, inclusive="both")
    & ~df["RefersToDocumentYear"].between(1900, 2100, inclusive="both")
    & df["RefersToDocumentYear"].notna()
)

df.loc[mask, ["RefersToDocumentNumber", "RefersToDocumentYear"]] = (
    df.loc[mask, ["RefersToDocumentYear", "RefersToDocumentNumber"]].to_numpy()
)


In [4]:
removers = df.loc[
    df["DocumentType"].isin(["R", "T"]) & df["RefersToDocumentNumber"].notna()
].copy()


In [5]:
audit_df = removers.merge(
    df,
    left_on="RefersToDocumentNumber",
    right_on="DocumentNumber",
    how="left",
    suffixes=("_remover", "_removed")
)


audit_df["RemovedDocumentFound"] = audit_df["DocumentNumber_removed"].notna()

audit_df = audit_df[
    [
        "DocumentNumber_remover",
        "DocumentType_remover",
        "RefersToDocumentNumber_remover",
        "RefersToDocumentYear_remover",
        "DocumentNumber_removed",
        "DocumentType_removed",
        "RemovedDocumentFound",
    ]
].rename(columns={
    "DocumentNumber_remover": "RemoverDocumentNumber",
    "DocumentType_remover": "RemoverDocumentType",
    "RefersToDocumentNumber_remover": "ReferencedDocumentNumber",
    "RefersToDocumentYear_remover": "ReferencedDocumentYear",
    "DocumentNumber_removed": "RemovedDocumentNumber",
    "DocumentType_removed": "RemovedDocumentType",
})


In [6]:
remove_docs = audit_df.loc[
    audit_df["RemovedDocumentFound"], "RemovedDocumentNumber"
].dropna().unique()


In [7]:
cleaned_df = df[~df["DocumentNumber"].isin(remove_docs)].copy()
cleaned_df = cleaned_df.reset_index(drop=True)


In [8]:
print("Rows fixed for swapped year/number:", mask.sum())
print("Documents to remove:", len(remove_docs))
print("Rows in audit table:", len(audit_df))


Rows fixed for swapped year/number: 40
Documents to remove: 33
Rows in audit table: 54


In [9]:
cleaned_df


,DocumentNumber,DocumentDate,DocumentType,RefersToDocumentNumber,RefersToDocumentYear,Remarks
0,27,8/16/2021,C,NaN,NaN,NaN
1,67,10/9/2020,A,NaN,NaN,NaN
2,157,10/9/2020,A,NaN,NaN,NaN
3,189,10/9/2020,J,NaN,NaN,NaN
4,250,10/9/2020,R,98.0,2016.0,NaN
...,...,...,...,...,...,...
90,36,4/16/1991,R,21.0,1991.0,NaN
91,2,11/15/1990,B,NaN,NaN,NaN
92,10,7/18/1988,B,NaN,NaN,NaN
93,76,8/28/1987,R,NaN,NaN,NaN


In [10]:
audit_df

,RemoverDocumentNumber,RemoverDocumentType,ReferencedDocumentNumber,ReferencedDocumentYear,RemovedDocumentNumber,RemovedDocumentType,RemovedDocumentFound
0,1,T,8.0,2020.0,8,B,True
1,250,R,98.0,2016.0,98,A,True
2,87,R,97.0,1992.0,97,B,True
3,87,R,97.0,1992.0,97,R,True
4,87,R,97.0,1992.0,97,A,True
5,117,R,84.0,2013.0,84,B,True
6,59,R,199.0,2002.0,199,B,True
7,82,R,130.0,2015.0,130,A,True
8,212,R,100.0,2010.0,100,B,True
9,212,R,100.0,2010.0,100,C,True
